In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load your data
path = "PredictorData2024.xlsx"
gw = pd.read_excel(path, sheet_name="Monthly", header=0)

# Date handling
gw['yyyymm'] = gw['yyyymm'].astype(int)
gw['DATE'] = pd.to_datetime(gw['yyyymm'].astype(str), format='%Y%m')
gw = gw.set_index('DATE').sort_index()

# --- Build predictors (contemporaneous) ---
X = pd.DataFrame(index=gw.index)
X['dp'] = np.log(gw['D12']) - np.log(gw['Index'])
X['dy'] = np.log(gw['D12']) - np.log(gw['Index'].shift(1)) 
X['ep'] = np.log(gw['E12']) - np.log(gw['Index'])
X['de'] = np.log(gw['D12']) - np.log(gw['E12'])
X['bm'] = gw['b/m']
X['ntis'] = gw['ntis']
X['tbl'] = gw['tbl']
X['lty'] = gw['lty']
X['ltr'] = gw['ltr']
X['tms'] = gw['lty'] - gw['tbl']
X['dfy'] = gw['BAA'] - gw['AAA']
X['dfr'] = gw['corpr'] - gw['ltr']
X['svar'] = gw['svar']
X['infl'] = gw['infl']

# Dependent variable: excess return
y = gw['CRSP_SPvw'] - np.log(1+gw['Rfree'])
y.name = "excess_ret"

# --- Lag ALL predictors by 1 month ---
X_pred = X.shift(1)

# Restrict to 1927 and later
y = y.loc["1927-01-01":]
X_pred = X_pred.loc["1927-01-01":]


# Final predictive dataset
data = pd.concat([y, X_pred], axis=1)










c:\Users\au506673\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [ ]:
#Data Analysis

# Summary statistics
summary_stats = data[['dp','dy','bm','ntis','excess_ret']].describe().T[['mean','std','min','max']]
print(summary_stats)


#Plotting y and selection x variables for visual check

import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(14,10))

# Top: Excess returns
axs[0].plot(data.index, data['excess_ret'], color='black')
axs[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axs[0].set_title("Excess Market Return (y)")
axs[0].set_ylabel("Return")

# Bottom: Four predictors
axs[1].plot(data.index, data['dp'], label='dp')
axs[1].plot(data.index, data['ep'], label='ep')
axs[1].plot(data.index, data['bm'], label='bm')
axs[1].plot(data.index, data['ntis'], label='ntis')
axs[1].set_title("Selected Predictors")
axs[1].legend()
axs[1].set_ylabel("Value")

plt.tight_layout()
plt.show()


In [ ]:
import statsmodels.api as sm

# ==========================
# Single-predictor regressions
# ==========================
single_pred_results = []

for col in X_pred.columns:
    X_single = sm.add_constant(X_pred[col])  # add intercept
    model = sm.OLS(y, X_single).fit()
    
    single_pred_results.append({
        'Predictor': col,
        'Coef': model.params[col],
        't_stat': model.tvalues[col],
        'p_value': model.pvalues[col],
        'IS_R2_percent': model.rsquared * 100
    })

# Convert to DataFrame and sort by R²
single_pred_df = pd.DataFrame(single_pred_results).sort_values('IS_R2_percent', ascending=False)

print("=== Single-Predictor Regressions ===")
single_pred_df


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

# ==========================
# Multivariate regressions
# ==========================

# Historical mean (benchmark)
mean_pred_in = np.repeat(y.mean(), len(y))
R2_mean = 1 - ((y - mean_pred_in)**2).sum() / ((y - y.mean())**2).sum()
R2_mean *= 100

# OLS (all predictors)
ols_in = LinearRegression().fit(X_pred, y)
ols_pred_in = ols_in.predict(X_pred)
R2_ols = 1 - ((y - ols_pred_in)**2).sum() / ((y - y.mean())**2).sum()
R2_ols *= 100

# TimeSeriesSplit for monthly returns
tscv = TimeSeriesSplit(n_splits=5)

# Lasso pipeline with time-series CV
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=tscv))
])
lasso_pipe.fit(X_pred, y)
lasso_pred_in = lasso_pipe.predict(X_pred)

# R² in-sample
R2_lasso = 1 - ((y - lasso_pred_in)**2).sum() / ((y - y.mean())**2).sum()
R2_lasso *= 100

# Post-Lasso
lasso_model = lasso_pipe.named_steps['lasso']
selected = np.where(lasso_model.coef_ != 0)[0]

if len(selected) > 0:
    ols_post_in = LinearRegression().fit(X_pred.iloc[:, selected], y)
    post_pred_in = ols_post_in.predict(X_pred.iloc[:, selected])
else:
    post_pred_in = np.repeat(y.mean(), len(y))

R2_post = 1 - ((y - post_pred_in)**2).sum() / ((y - y.mean())**2).sum()
R2_post *= 100

# Combine results in a DataFrame
multivariate_df = pd.DataFrame({
    'Model': ['HistoricalMean', 'OLS', 'Lasso', 'Post-Lasso'],
    'IS_R2_percent': [R2_mean, R2_ols, R2_lasso, R2_post]
})

print("=== Multivariate Regressions ===")
multivariate_df

In [ ]:
#Function for Rolling out-of-sample Forecasts

def rolling_forecast(y, X, window=None):
    mean_preds, ols_preds, lasso_preds, post_preds, y_true, forecast_dates = [], [], [], [], [], []
    dates = X.index

    # TimeSeriesSplit for LassoCV within each rolling window
    tscv = TimeSeriesSplit(n_splits=5)

    for t in range(window, len(dates)):
        test_date = dates[t]

        # Training = last `window` months
        train_idx = dates[t-window:t]
        y_train, X_train = y.loc[train_idx], X.loc[train_idx]
        y_test, X_test = y.loc[test_date], X.loc[[test_date]]

        # --- Historical mean ---
        mean_pred = y_train.mean()

        # --- OLS ---
        ols = LinearRegression().fit(X_train, y_train)
        ols_pred = ols.predict(X_test)[0]

        # --- Lasso with TimeSeriesSplit ---
        lasso_pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('lasso', LassoCV(cv=tscv))
        ])
        lasso_pipe.fit(X_train, y_train)
        lasso_pred = lasso_pipe.predict(X_test)[0]

        # --- Post-Lasso ---
        lasso_model = lasso_pipe.named_steps['lasso']
        selected = np.where(lasso_model.coef_ != 0)[0]

        if len(selected) > 0:
            ols_post = LinearRegression().fit(X_train.iloc[:, selected], y_train)
            post_pred = ols_post.predict(X_test.iloc[:, selected])[0]
        else:
            post_pred = mean_pred  # fallback

        # Store results
        mean_preds.append(mean_pred)
        ols_preds.append(ols_pred)
        lasso_preds.append(lasso_pred)
        post_preds.append(post_pred)
        y_true.append(y_test)
        forecast_dates.append(test_date)

    # Return DataFrame
    return pd.DataFrame({
        "HistoricalMean": mean_preds,
        "OLS": ols_preds,
        "Lasso": lasso_preds,
        "PostLasso": post_preds,
        "y_true": y_true
    }, index=forecast_dates)

# Example usage
res_oos = rolling_forecast(y, X_pred, window=240)
print(res_oos.head())


In [ ]:
# Use the output from your rolling_forecast function

# Rolling window lengths in months
window_lengths = [240, 360, 480]  #20, 30, 40 years

# Store results
oos_results_all = []

for w in window_lengths:
    print(f"Running rolling forecast with {w} months (~{w//12} years) window...")
    res_oos = rolling_forecast(y, X_pred, window=w)
    
    # Benchmark MSE
    y_true = res_oos["y_true"]
    benchmark_pred = res_oos["HistoricalMean"]
    mse_benchmark = ((y_true - benchmark_pred) ** 2).mean()
    
    models = ["HistoricalMean", "OLS", "Lasso", "PostLasso"]
    for model in models:
        y_pred = res_oos[model]
        mse_model = ((y_true - y_pred) ** 2).mean()
        R2_oos = 1 - mse_model / mse_benchmark
        R2_oos *= 100
        
        oos_results_all.append({
            "Window_months": w,
            "Window_years": w//12,
            "Model": model,
            "OOS_MSE": mse_model,
            "OOS_R2_percent": R2_oos
        })

# Convert to DataFrame
oos_df_windows = pd.DataFrame(oos_results_all)
oos_df_windows = oos_df_windows.sort_values(["Window_months", "OOS_R2_percent"], ascending=[True, False])
oos_df_windows






In [ ]:
#Which variables are selected in forecasting across time?

# Choose a rolling window length
window = 360  # 30 years
dates = X_pred.index

# Initialize selection matrix
selection_matrix = pd.DataFrame(0, index=dates[window:], columns=X_pred.columns)

# Rolling Lasso with TimeSeriesSplit
for t in range(window, len(dates)):
    train_idx = dates[t-window:t]
    X_train, y_train = X_pred.loc[train_idx], y.loc[train_idx]
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # TimeSeriesSplit (e.g., 5 splits)
    tscv = TimeSeriesSplit(n_splits=5)
    
    # LassoCV with time series splits
    lasso = LassoCV(cv=tscv).fit(X_train_scaled, y_train)
    
    # Track selected variables
    selected = X_train.columns[lasso.coef_ != 0]
    selection_matrix.loc[dates[t], selected] = 1

# Plot selection matrix
plt.figure(figsize=(12,6))
for i, var in enumerate(selection_matrix.columns):
    sel_dates = selection_matrix.index[selection_matrix[var]==1]
    y_vals = np.ones(len(sel_dates)) * i   # <-- use np.ones(len(...)) instead of np.ones_like
    plt.scatter(sel_dates, y_vals, color="blue", s=15)

plt.yticks(range(len(selection_matrix.columns)), selection_matrix.columns)
plt.xlabel("Date")
plt.ylabel("Predictors")
plt.title(f"Lasso Variable Selection Over Time (Window = {window//12} years, TimeSeriesCV)")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


